## SpendWise AI - Notebook 06: LLM Integration

**Objective:** Build natural language interface using Claude API

**Features:**
- Query spending data in plain English
- Get insights and recommendations
- Connect all models (classifier, anomaly, forecaster)

**What you'll learn:**
- Anthropic Claude API usage
- Function/Tool calling pattern
- Building conversational AI for finance

### 1. Imports & Setup

In [1]:
import json
import os
from pathlib import Path
from datetime import datetime, timedelta
from typing import Optional

import numpy as np
import pandas as pd

try:
    from anthropic import Anthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    ANTHROPIC_AVAILABLE = False
    print("anthropic not installed. Run: pip install anthropic")

from dataclasses import dataclass
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "src" and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists():
    while PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent
        if (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "src").exists():
            break
print("PROJECT_ROOT:", PROJECT_ROOT)

anthropic not installed. Run: pip install anthropic
PROJECT_ROOT: /Users/patel/Documents/E/MPS Sem 5/EAI 6020/Final_Project/SpendwiseAI


### 2. Load Transaction Data & Models

In [2]:
transactions_df = pd.read_csv(PROJECT_ROOT / "data/synthetic/transactions_full.csv")
transactions_df["date"] = pd.to_datetime(transactions_df["date"])
with open(PROJECT_ROOT / "data/processed/label_mappings.json", "r") as f:
    label_mappings = json.load(f)
print(f"Data loaded: {len(transactions_df):,} transactions")
print(f"Date range: {transactions_df['date'].min().date()} to {transactions_df['date'].max().date()}")

Data loaded: 122,754 transactions
Date range: 2025-09-15 to 2026-03-13


### 3. Build Data Query Functions

These functions retrieve data based on user queries. The LLM will call them to get information.

In [3]:
class FinancialDataManager:
    """Manages financial data queries for the LLM."""

    def __init__(self, transactions_df: pd.DataFrame):
        self.df = transactions_df.copy()
        self.df["date"] = pd.to_datetime(self.df["date"])
        self.df["month"] = self.df["date"].dt.to_period("M")
        self.df["week"] = self.df["date"].dt.to_period("W")
        self.df["is_expense"] = self.df["amount"] < 0

    def get_spending_by_category(self, user_id: str, start_date: str = None, end_date: str = None) -> dict:
        df = self._filter_user_and_dates(user_id, start_date, end_date)
        expenses = df[df["is_expense"]].copy()
        expenses["amount"] = expenses["amount"].abs()
        by_category = expenses.groupby("category")["amount"].sum().sort_values(ascending=False)
        return {
            "user_id": user_id,
            "period": f"{start_date or 'all time'} to {end_date or 'now'}",
            "total_spending": round(expenses["amount"].sum(), 2),
            "by_category": {cat: round(amt, 2) for cat, amt in by_category.items()},
            "top_category": by_category.index[0] if len(by_category) > 0 else None,
        }

    def get_spending_trend(self, user_id: str, category: str = None, period: str = "monthly") -> dict:
        df = self._filter_user_and_dates(user_id)
        expenses = df[df["is_expense"]].copy()
        expenses["amount"] = expenses["amount"].abs()
        if category:
            expenses = expenses[expenses["category"] == category]
        grouped = expenses.groupby("week")["amount"].sum() if period == "weekly" else expenses.groupby("month")["amount"].sum()
        trend = {str(k): round(v, 2) for k, v in grouped.tail(6).to_dict().items()}
        values = list(trend.values())
        change = ((values[-1] - values[-2]) / values[-2]) * 100 if len(values) >= 2 and values[-2] > 0 else 0
        return {
            "user_id": user_id, "category": category or "all", "period": period,
            "trend": trend, "latest_change_percent": round(change, 1),
            "average": round(np.mean(values), 2) if values else 0,
        }

    def get_subscriptions(self, user_id: str) -> dict:
        df = self._filter_user_and_dates(user_id)
        expenses = df[df["is_expense"]].copy()
        expenses["amount"] = expenses["amount"].abs()
        merchant_counts = expenses.groupby("merchant").agg({"amount": ["count", "mean", "std"], "date": ["min", "max"]})
        merchant_counts.columns = ["count", "avg_amount", "std_amount", "first_date", "last_date"]
        subscriptions = merchant_counts[(merchant_counts["count"] >= 3) & (merchant_counts["std_amount"] < 1)].copy()
        subscriptions["monthly_cost"] = subscriptions["avg_amount"]
        sub_list = [{"merchant": m, "monthly_cost": round(row["monthly_cost"], 2), "occurrences": int(row["count"])} for m, row in subscriptions.iterrows()]
        total_monthly = sum(s["monthly_cost"] for s in sub_list)
        return {"user_id": user_id, "subscriptions": sorted(sub_list, key=lambda x: x["monthly_cost"], reverse=True), "total_monthly": round(total_monthly, 2), "total_yearly": round(total_monthly * 12, 2), "count": len(sub_list)}

    def get_spending_summary(self, user_id: str, period: str = "last_month") -> dict:
        max_date = self.df["date"].max()
        start = max_date - timedelta(days=30 if period == "last_month" else 7 if period == "last_week" else 90) if period in ("last_month", "last_week", "last_3_months") else None
        start_str = start.strftime("%Y-%m-%d") if start is not None else None
        category_data = self.get_spending_by_category(user_id, start_str)
        trend_data = self.get_spending_trend(user_id)
        df = self._filter_user_and_dates(user_id, start_str)
        n_transactions = len(df[df["is_expense"]])
        return {
            "user_id": user_id, "period": period, "total_spending": category_data["total_spending"],
            "transaction_count": n_transactions, "top_categories": dict(list(category_data["by_category"].items())[:5]),
            "spending_change": trend_data["latest_change_percent"],
            "daily_average": round(category_data["total_spending"] / 30, 2) if period == "last_month" else None,
        }

    def compare_to_average(self, user_id: str, category: str) -> dict:
        max_date = self.df["date"].max()
        last_month_start = (max_date - timedelta(days=30)).strftime("%Y-%m-%d")
        recent = self.get_spending_by_category(user_id, last_month_start)
        recent_amount = recent["by_category"].get(category, 0)
        all_time = self.get_spending_by_category(user_id)
        df = self._filter_user_and_dates(user_id)
        n_months = df["month"].nunique()
        historical_avg = all_time["by_category"].get(category, 0) / max(n_months, 1)
        diff_percent = ((recent_amount - historical_avg) / historical_avg) * 100 if historical_avg > 0 else 0
        return {"user_id": user_id, "category": category, "recent_spending": round(recent_amount, 2), "historical_monthly_avg": round(historical_avg, 2), "difference_percent": round(diff_percent, 1), "status": "above" if diff_percent > 10 else "below" if diff_percent < -10 else "normal"}

    def _filter_user_and_dates(self, user_id: str, start_date: str = None, end_date: str = None) -> pd.DataFrame:
        df = self.df[self.df["user_id"] == user_id]
        if start_date:
            df = df[df["date"] >= start_date]
        if end_date:
            df = df[df["date"] <= end_date]
        return df

In [4]:
data_manager = FinancialDataManager(transactions_df)
test_user = "user_0001"
print("Testing Data Functions:\n")
result = data_manager.get_spending_by_category(test_user)
print(f"1. Spending by category: Total ${result['total_spending']:.2f}, Top 3: {dict(list(result['by_category'].items())[:3])}")
subs = data_manager.get_subscriptions(test_user)
print(f"2. Subscriptions: {subs['count']}, Monthly total ${subs['total_monthly']:.2f}")
summary = data_manager.get_spending_summary(test_user, "last_month")
print(f"3. Last month: Total ${summary['total_spending']:.2f}, Transactions {summary['transaction_count']}")

Testing Data Functions:

1. Spending by category: Total $92912.60, Top 3: {'Financial': 20585.83, 'Travel': 18574.5, 'Education': 17471.25}
2. Subscriptions: 1, Monthly total $8.77
3. Last month: Total $18112.45, Transactions 134


### 4. Define Tools for Claude

Tools tell Claude what functions are available and how to call them.

In [5]:
TOOLS = [
    {"name": "get_spending_by_category", "description": "Get total spending by category for a time period.", "input_schema": {"type": "object", "properties": {"user_id": {"type": "string"}, "start_date": {"type": "string"}, "end_date": {"type": "string"}}, "required": ["user_id"]}},
    {"name": "get_spending_trend", "description": "Get spending trend over time (weekly or monthly).", "input_schema": {"type": "object", "properties": {"user_id": {"type": "string"}, "category": {"type": "string"}, "period": {"type": "string", "enum": ["weekly", "monthly"]}}, "required": ["user_id"]}},
    {"name": "get_subscriptions", "description": "Detect recurring charges/subscriptions.", "input_schema": {"type": "object", "properties": {"user_id": {"type": "string"}}, "required": ["user_id"]}},
    {"name": "get_spending_summary", "description": "Get spending summary for a period.", "input_schema": {"type": "object", "properties": {"user_id": {"type": "string"}, "period": {"type": "string", "enum": ["last_week", "last_month", "last_3_months"]}}, "required": ["user_id"]}},
    {"name": "compare_to_average", "description": "Compare recent spending in a category to historical average.", "input_schema": {"type": "object", "properties": {"user_id": {"type": "string"}, "category": {"type": "string"}}, "required": ["user_id", "category"]}},
]
print(f"Defined {len(TOOLS)} tools for Claude")

Defined 5 tools for Claude


### 5. Build the Financial Assistant Class

In [6]:
class FinancialAssistant:
    """AI-powered financial assistant using Claude."""

    def __init__(self, data_manager: FinancialDataManager, api_key: str = None):
        self.data_manager = data_manager
        self.api_key = api_key or os.environ.get("ANTHROPIC_API_KEY")
        if self.api_key and ANTHROPIC_AVAILABLE:
            self.client = Anthropic(api_key=self.api_key)
            self.use_api = True
        else:
            self.client = None
            self.use_api = False
            print("Running in demo mode (no API key). Responses will be simulated.")
        self.system_prompt = "You are a helpful financial assistant for SpendWise AI. Be concise; use numbers from the data; format currency with $ and 2 decimal places. Categories: Food & Dining, Transportation, Shopping, Bills & Utilities, Subscriptions, Entertainment, Health & Wellness, Travel, Education, Personal Care, Financial, Income"
        self.tool_functions = {"get_spending_by_category": self.data_manager.get_spending_by_category, "get_spending_trend": self.data_manager.get_spending_trend, "get_subscriptions": self.data_manager.get_subscriptions, "get_spending_summary": self.data_manager.get_spending_summary, "compare_to_average": self.data_manager.compare_to_average}

    def chat(self, user_message: str, user_id: str = "user_0001") -> str:
        full_message = f"[User ID: {user_id}] {user_message}"
        return self._chat_with_api(full_message, user_id) if self.use_api else self._chat_demo(user_message, user_id)

    def _chat_with_api(self, message: str, user_id: str) -> str:
        messages = [{"role": "user", "content": message}]
        response = self.client.messages.create(model="claude-sonnet-4-20250514", max_tokens=1024, system=self.system_prompt, tools=TOOLS, messages=messages)
        while response.stop_reason == "tool_use":
            tool_use_block = next((b for b in response.content if getattr(b, "type", None) == "tool_use"), None)
            if not tool_use_block:
                break
            tool_input = dict(tool_use_block.input)
            if "user_id" not in tool_input:
                tool_input["user_id"] = user_id
            tool_result = self.tool_functions[tool_use_block.name](**tool_input)
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_use_block.id, "content": json.dumps(tool_result)}]})
            response = self.client.messages.create(model="claude-sonnet-4-20250514", max_tokens=1024, system=self.system_prompt, tools=TOOLS, messages=messages)
        for block in response.content:
            if hasattr(block, "text"):
                return block.text
        return "I couldn't generate a response. Please try again."

    def _chat_demo(self, message: str, user_id: str) -> str:
        msg = message.lower()
        if any(w in msg for w in ["subscription", "recurring", "monthly"]):
            r = self.data_manager.get_subscriptions(user_id)
            top = "\n".join([f"  {s['merchant']}: ${s['monthly_cost']:.2f}/month" for s in r["subscriptions"][:3]])
            return f"Found {r['count']} recurring charges:\n{top}\nTotal monthly: ${r['total_monthly']:.2f} (${r['total_yearly']:.2f}/year)"
        if any(w in msg for w in ["category", "breakdown", "where", "spent"]):
            r = self.data_manager.get_spending_by_category(user_id)
            top = "\n".join([f"  {c}: ${a:.2f}" for c, a in list(r["by_category"].items())[:5]])
            return f"Spending breakdown:\n{top}\nTotal: ${r['total_spending']:.2f}. Top category: {r['top_category']}"
        if any(w in msg for w in ["trend", "pattern", "change", "over time"]):
            r = self.data_manager.get_spending_trend(user_id, period="monthly")
            d = "increased" if r["latest_change_percent"] > 0 else "decreased"
            return f"Spending {d} by {abs(r['latest_change_percent']):.1f}%. Average: ${r['average']:.2f}. Recent: {r['trend']}"
        if any(w in msg for w in ["summary", "overview", "total", "how much"]):
            r = self.data_manager.get_spending_summary(user_id, "last_month")
            return f"Last month: ${r['total_spending']:.2f}, {r['transaction_count']} txns, daily avg ${r['daily_average']:.2f}, change {r['spending_change']:+.1f}%. Top: {list(r['top_categories'].items())[:3]}"
        return "Try: 'What are my subscriptions?', 'Show spending by category', 'Spending trend?', 'Summary of last month', 'Compare food spending to average?'"

### 6. Test the Assistant

In [7]:
assistant = FinancialAssistant(data_manager)
print("Financial Assistant ready.\n" + "=" * 60)
for query in ["What are my subscriptions?", "Show me my spending by category", "What's my spending trend?", "Give me a summary of last month"]:
    print(f"\nUser: {query}\nAssistant:\n{assistant.chat(query, user_id='user_0001')}\n" + "-" * 60)

Running in demo mode (no API key). Responses will be simulated.
Financial Assistant ready.

User: What are my subscriptions?
Assistant:
Found 1 recurring charges:
  VANCOUVER TRANSIT: $8.77/month
Total monthly: $8.77 ($105.24/year)
------------------------------------------------------------

User: Show me my spending by category
Assistant:
Spending breakdown:
  Financial: $20585.83
  Travel: $18574.50
  Education: $17471.25
  Health & Wellness: $7765.42
  Food & Dining: $7112.85
Total: $92912.60. Top category: Financial
------------------------------------------------------------

User: What's my spending trend?
Assistant:
Spending decreased by 46.7%. Average: $14267.91. Recent: {'2025-10': 12263.23, '2025-11': 17852.07, '2025-12': 17714.09, '2026-01': 15087.17, '2026-02': 14799.77, '2026-03': 7891.12}
------------------------------------------------------------

User: Give me a summary of last month
Assistant:
Last month: $18112.45, 134 txns, daily avg $603.75, change -46.7%. Top: [(

### 7. Interactive Chat Interface (for Streamlit later)

In [8]:
class ChatSession:
    """Maintains conversation history for multi-turn chat."""
    def __init__(self, assistant: FinancialAssistant, user_id: str = "user_0001"):
        self.assistant = assistant
        self.user_id = user_id
        self.history = []
    def send(self, message: str) -> str:
        response = self.assistant.chat(message, self.user_id)
        self.history.append({"role": "user", "content": message, "timestamp": datetime.now().isoformat()})
        self.history.append({"role": "assistant", "content": response, "timestamp": datetime.now().isoformat()})
        return response
    def get_history(self):
        return self.history
    def clear(self):
        self.history = []

session = ChatSession(assistant, user_id="user_0001")
for msg in ["Hi! What can you help me with?", "Show me my subscriptions", "What about my food spending?"]:
    print(f"You: {msg}\nSpendWise: {session.send(msg)}\n" + "-" * 40)

You: Hi! What can you help me with?
SpendWise: Try: 'What are my subscriptions?', 'Show spending by category', 'Spending trend?', 'Summary of last month', 'Compare food spending to average?'
----------------------------------------
You: Show me my subscriptions
SpendWise: Found 1 recurring charges:
  VANCOUVER TRANSIT: $8.77/month
Total monthly: $8.77 ($105.24/year)
----------------------------------------
You: What about my food spending?
SpendWise: Try: 'What are my subscriptions?', 'Show spending by category', 'Spending trend?', 'Summary of last month', 'Compare food spending to average?'
----------------------------------------


### 8. Add Anomaly & Forecast Integration

In [9]:
class EnhancedFinancialAssistant(FinancialAssistant):
    """Enhanced assistant with anomaly detection and forecasting."""

    def __init__(self, data_manager, api_key=None):
        super().__init__(data_manager, api_key)
        self.anomaly_detector = None
        self.forecaster = None
        if (PROJECT_ROOT / "models/anomaly_model/model.pt").exists():
            try:
                import sys
                sys.path.insert(0, str(PROJECT_ROOT / "src"))
                from anomaly_detector import AnomalyDetector
                self.anomaly_detector = AnomalyDetector(str(PROJECT_ROOT / "models/anomaly_model"))
                print("Anomaly detector loaded")
            except Exception as e:
                print("Could not load anomaly detector:", e)
        if (PROJECT_ROOT / "models/forecaster_model/model.pt").exists():
            try:
                import sys
                sys.path.insert(0, str(PROJECT_ROOT / "src"))
                from spending_forecaster import ZICATTInference
                self.forecaster = ZICATTInference(str(PROJECT_ROOT / "models/forecaster_model"))
                print("Spending forecaster loaded")
            except Exception as e:
                print("Could not load forecaster:", e)

    def check_anomalies(self, user_id: str) -> dict:
        if not self.anomaly_detector:
            return {"error": "Anomaly detector not available"}
        result = self.data_manager.get_spending_by_category(user_id)
        det = self.anomaly_detector.detect(result["by_category"])
        return {"user_id": user_id, "anomaly_score": det["anomaly_score"], "is_anomaly": det["is_anomaly"], "status": "unusual" if det["is_anomaly"] else "normal"}

    def forecast_spending(self, user_id: str) -> dict:
        if not self.forecaster:
            return {"error": "Forecaster not available"}
        trend = self.data_manager.get_spending_trend(user_id, period="weekly")
        history = list(trend["trend"].values())
        if len(history) < 8:
            return {"error": "Not enough history for forecast (need 8 weeks)"}
        pred = self.forecaster.predict(history)
        return {"user_id": user_id, "predicted_spending": pred["predicted_spending"], "range": f"${pred['lower_bound']} - ${pred['upper_bound']}", "recent_average": round(np.mean(history[-4:]), 2)}

### 9. Test Enhanced Assistant

In [10]:
enhanced_assistant = EnhancedFinancialAssistant(data_manager)
print("Testing Enhanced Features:\n")
ar = enhanced_assistant.check_anomalies("user_0001")
print("1. Anomaly Detection:", ar.get("status", ar.get("error", "N/A")), f"Score: {ar.get('anomaly_score', 0):.1f}/100" if "error" not in ar else "")
fr = enhanced_assistant.forecast_spending("user_0001")
if "error" not in fr:
    print("2. Forecast: ${:.2f}, Range: {}, Recent avg ${:.2f}".format(fr["predicted_spending"], fr["range"], fr["recent_average"]))
else:
    print("2. Forecast:", fr["error"])

Running in demo mode (no API key). Responses will be simulated.
Anomaly detector loaded
Spending forecaster loaded
Testing Enhanced Features:

1. Anomaly Detection: unusual Score: 100.0/100
2. Forecast: Not enough history for forecast (need 8 weeks)


### 10. Save the LLM Module

In [11]:
try:
    import nbformat
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nbformat"])
    import nbformat

if "PROJECT_ROOT" not in dir():
    from pathlib import Path
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "src" and (PROJECT_ROOT.parent / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif not (PROJECT_ROOT / "data").exists():
        while PROJECT_ROOT != PROJECT_ROOT.parent:
            PROJECT_ROOT = PROJECT_ROOT.parent
            if (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "src").exists():
                break

notebook_path = PROJECT_ROOT / "src" / "06_llm_integration.ipynb"
nb = nbformat.read(str(notebook_path), as_version=4)

code_snippets = []
for cell in nb.cells:
    if cell.get("cell_type") == "code":
        src = cell.get("source", "")
        if isinstance(src, list):
            src = "".join(src)
        if (
            "class FinancialDataManager" in src
            or "TOOLS =" in src
            or "class FinancialAssistant" in src
            or "class ChatSession" in src
            or "class EnhancedFinancialAssistant" in src
        ):
            code_snippets.append(src)

module_code = """import json
import os
from datetime import datetime, timedelta
from typing import Optional

import numpy as np
import pandas as pd
from dataclasses import dataclass

try:
    from anthropic import Anthropic
    ANTHROPIC_AVAILABLE = True
except Exception:
    Anthropic = None
    ANTHROPIC_AVAILABLE = False
"""

module_code += "\n\n".join(code_snippets) + "\n"

module_path = PROJECT_ROOT / "src" / "llm_assistant.py"
module_path.parent.mkdir(parents=True, exist_ok=True)
with open(module_path, "w") as f:
    f.write(module_code)
print(f"Module saved to {module_path}")

Module saved to /Users/patel/Documents/E/MPS Sem 5/EAI 6020/Final_Project/SpendwiseAI/src/llm_assistant.py


### Summary

- FinancialDataManager: query functions (spending by category, trend, subscriptions, summary, compare to average).
- Claude tools (TOOLS) and FinancialAssistant with API + demo mode; ChatSession for history.
- EnhancedFinancialAssistant: anomaly detector and forecaster integration.
- Production module: `src/llm_assistant.py`. Next: Notebook 07 (Recommendation Engine).

In [12]:
# Module saved by the cell under '10. Save the LLM Module'


In [13]:
try:
    import nbformat
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nbformat"])
    import nbformat

if "PROJECT_ROOT" not in dir():
    from pathlib import Path
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "src" and (PROJECT_ROOT.parent / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif not (PROJECT_ROOT / "data").exists():
        while PROJECT_ROOT != PROJECT_ROOT.parent:
            PROJECT_ROOT = PROJECT_ROOT.parent
            if (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "src").exists():
                break

notebook_path = PROJECT_ROOT / "src" / "06_llm_integration.ipynb"
nb = nbformat.read(str(notebook_path), as_version=4)

code_snippets = []
for cell in nb.cells:
    if cell.get("cell_type") == "code":
        src = cell.get("source", "")
        if isinstance(src, list):
            src = "".join(src)
        if (
            "class FinancialDataManager" in src
            or "TOOLS =" in src
            or "class FinancialAssistant" in src
        ):
            code_snippets.append(src)

module_code = '''import json
import os
from datetime import datetime, timedelta
from typing import Optional

import numpy as np
import pandas as pd
from dataclasses import dataclass

try:
    from anthropic import Anthropic
    ANTHROPIC_AVAILABLE = True
except Exception:
    Anthropic = None
    ANTHROPIC_AVAILABLE = False
'''

module_code += "\n\n".join(code_snippets) + "\n"

module_path = PROJECT_ROOT / "src" / "llm_assistant.py"
module_path.parent.mkdir(parents=True, exist_ok=True)
with open(module_path, "w") as f:
    f.write(module_code)
print(f"Module saved to {module_path}")

Module saved to /Users/patel/Documents/E/MPS Sem 5/EAI 6020/Final_Project/SpendwiseAI/src/llm_assistant.py
